# xBD Damage Segmentation: full reproducible experiment

This notebook reproduces the full experiment end-to-end:
1. Train 4 segmentation models (U-Net, BAFUNet, FusionUNet, ResUNet++) with shared ResNet34 ImageNet encoder
2. Run BAFUNet ablation over `aggre_depth ∈ {1, 2, 3}`
3. Evaluate all on test split
4. Build comparison tables, training curves, prediction grids, error analysis

**Prerequisite**: dataset is already prepared via `python -m src.data.prepare_xbd`.

Resume is supported: rerunning a cell after interruption picks up from the latest checkpoint.

## Setup

In [ ]:
import os, sys, subprocess
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print(f"Working dir: {Path.cwd()}")

# Verify dataset is ready
manifest = Path("xbd_processed/manifest.json")
if not manifest.exists():
    raise SystemExit(
        "Dataset not found. Run preprocessing first:\n"
        "  python -m src.data.prepare_xbd --xbd-root ./train --output ./xbd_processed"
    )
print(f"Dataset OK: {manifest}")

## Run Helper

In [ ]:
def run(cmd):
    print(f"\n>>> {' '.join(cmd)}")
    r = subprocess.run(cmd, cwd=str(ROOT))
    if r.returncode != 0:
        raise SystemExit(f"Command failed: {cmd}")

## Main Training

In [ ]:
MAIN_CONFIGS = [
    "configs/unet.yaml",
    "configs/bafunet.yaml",
    "configs/fusion_unet.yaml",
    "configs/resunet_pp.yaml",
]
for cfg in MAIN_CONFIGS:
    run(["python", "-m", "src.training.train", "--config", cfg, "--resume", "auto"])

## Ablation

In [ ]:
run(["python", "scripts/run_ablation.py"])

## Evaluate All

In [ ]:
ALL_CONFIGS = MAIN_CONFIGS + [
    f"configs/_ablation/bafunet_ablation_d{d}.yaml" for d in [1, 2, 3]
]
for cfg in ALL_CONFIGS:
    run(["python", "scripts/evaluate.py", "--config", cfg])

## Main Comparison Table

In [ ]:
import pandas as pd
import json

MAIN_RUNS = {
    "U-Net": "unet_resnet34",
    "BAFUNet": "bafunet_resnet34",
    "FusionUNet": "fusion_unet_resnet34",
    "ResUNet++": "resunet_pp_resnet34",
}
rows = []
for label, run_name in MAIN_RUNS.items():
    metrics = json.load(open(Path("checkpoints") / run_name / "test_metrics.json"))["metrics"]
    rows.append({"model": label, **{k: round(v, 4) for k, v in metrics.items()}})
df_main = pd.DataFrame(rows).sort_values("dice", ascending=False).reset_index(drop=True)
df_main.to_csv("results/main_comparison.csv", index=False)
df_main

## Training Curves Comparison

In [ ]:
from src.training.plots import plot_compare_runs

results_dir = Path("results"); results_dir.mkdir(exist_ok=True)
run_dirs = [Path("checkpoints") / r for r in MAIN_RUNS.values()]
plot_compare_runs(run_dirs, results_dir / "compare_val_dice.png", metric="dice", split="val",
                  title="Validation Dice across models")
plot_compare_runs(run_dirs, results_dir / "compare_val_loss.png", metric="loss", split="val",
                  title="Validation Loss across models")

from PIL import Image
import matplotlib.pyplot as plt
for p in [results_dir / "compare_val_dice.png", results_dir / "compare_val_loss.png"]:
    plt.figure(figsize=(10, 5))
    plt.imshow(Image.open(p)); plt.axis("off"); plt.show()

## Ablation Table

In [ ]:
ablation_rows = []
for d in [1, 2, 3]:
    rn = f"bafunet_ablation_d{d}"
    metrics = json.load(open(Path("checkpoints") / rn / "test_metrics.json"))["metrics"]
    ablation_rows.append({"aggre_depth": d, **{k: round(v, 4) for k, v in metrics.items()}})
df_abl = pd.DataFrame(ablation_rows)
df_abl.to_csv("results/ablation.csv", index=False)
df_abl

## Visualize Predictions for Best Model

In [ ]:
best_run = df_main.iloc[0]["model"]
best_cfg = {v: f"configs/{Path(v).name}" for v in MAIN_RUNS.values()}
# pick config matching best run
best_run_name = MAIN_RUNS[best_run]
best_config = next(c for c in MAIN_CONFIGS if best_run_name in str(c) or Path(c).stem in best_run_name)
# fallback by mapping
config_for_run = {
    "unet_resnet34": "configs/unet.yaml",
    "bafunet_resnet34": "configs/bafunet.yaml",
    "fusion_unet_resnet34": "configs/fusion_unet.yaml",
    "resunet_pp_resnet34": "configs/resunet_pp.yaml",
}
best_config = config_for_run[best_run_name]

run(["python", "scripts/visualize_predictions.py",
     "--config", best_config,
     "--output", "results/predictions_grid.png",
     "--n-samples", "12", "--strategy", "mixed"])

plt.figure(figsize=(12, 18))
plt.imshow(Image.open("results/predictions_grid.png")); plt.axis("off"); plt.show()

## Error Analysis

In [ ]:
for run_name in MAIN_RUNS.values():
    cfg = config_for_run[run_name]
    run(["python", "scripts/error_analysis.py",
         "--config", cfg,
         "--output-dir", f"results/errors_{run_name}"])

# Show histograms for best model
err_dir = Path(f"results/errors_{best_run_name}")
plt.figure(figsize=(8, 4))
plt.imshow(Image.open(err_dir / "dice_distribution.png")); plt.axis("off"); plt.show()
plt.figure(figsize=(8, 4))
plt.imshow(Image.open(err_dir / "dice_vs_damage_ratio.png")); plt.axis("off"); plt.show()

## Final Summary

In [ ]:
print("=" * 70)
print("FINAL SUMMARY")
print("=" * 70)
print("\nMain comparison (test split):")
print(df_main.to_string(index=False))
print("\nBAFUNet ablation:")
print(df_abl.to_string(index=False))
print("\nResults saved to: results/")
print("  - main_comparison.csv, ablation.csv")
print("  - compare_val_dice.png, compare_val_loss.png")
print("  - predictions_grid.png")
print("  - errors_<run>/dice_distribution.png, dice_vs_damage_ratio.png, by_disaster.csv")